# Module Sante - Preparation et harmonisation

Notebook de nettoyage et preparation du domaine Sante (Couverture, Epidemiologie, Infrastructures).
Les controles (apercu, describe, isnull) sont affiches directement dans les cellules.

## Sections

1. Chargement des fichiers et inventaire
2. Fonctions utilitaires de nettoyage
3. Nettoyage sous-domaine Couverture
4. Nettoyage sous-domaine Epidemiologie
5. Nettoyage sous-domaine Infrastructures
6. Nettoyage indicateurs nationaux (sante_afristat)
7. Fusion des datasets necessaires (niveau annuel)
8. Harmonisation finale et suppression de colonnes non importantes
9. Exports vers data/Sante et recapitulatif

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [ ]:
ROOT = Path.cwd()
RAW_DIR = ROOT / "data_raw" / "sante"
OUT_ROOT = ROOT / "data" / "Sante"
OUT_COUV = OUT_ROOT / "Couverture"
OUT_EPI = OUT_ROOT / "Epidemiologie"
OUT_INFRA = OUT_ROOT / "Infrastructures"
for d in [OUT_ROOT, OUT_COUV, OUT_EPI, OUT_INFRA]:
    d.mkdir(parents=True, exist_ok=True)

files = sorted([p for p in RAW_DIR.rglob("*.csv") if not p.name.startswith(".~lock")])
inventory = pd.DataFrame({
    "fichier": [str(p.relative_to(RAW_DIR)) for p in files],
    "taille_ko": [round(p.stat().st_size / 1024, 2) for p in files]
})
print("Apercu inventaire sante:")
display(inventory)

In [ ]:
action_logs = []

def log_action(section: str, action: str, details: str) -> None:
    action_logs.append({"section": section, "action": action, "details": details})

def to_snake(text: str) -> str:
    text = str(text).strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [to_snake(c) for c in out.columns]
    return out

def normalize_text(v):
    if pd.isna(v):
        return v
    return re.sub(r"\s+", " ", str(v).strip())

def clean_geo_name(v):
    if pd.isna(v):
        return v
    t = normalize_text(v)
    t = re.sub(r"\b(region|province)\b", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip(" -_,")
    return t.title() if t else pd.NA

def drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    before = len(df)
    cleaned = df.dropna(how="all").copy()
    return cleaned, before - len(cleaned)

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "colonne": df.columns,
        "nb_manquants": [int(df[c].isna().sum()) for c in df.columns],
        "pct_manquants": [round(float(df[c].isna().mean()) * 100, 2) for c in df.columns]
    })
    return rep.sort_values("pct_manquants", ascending=False)

KEEP_COLUMNS = [
    "domaine", "sous_domaine", "source_file", "indicateur",
    "categorie_1", "categorie_2", "unite", "annee", "valeur",
    "pays", "region", "milieu", "sexe"
]

def build_standard_table(df: pd.DataFrame, sous_domaine: str, source_file: str) -> pd.DataFrame:
    data = normalize_columns(df)
    data, removed = drop_fully_empty_rows(data)
    log_action(sous_domaine, "supprimer_lignes_vides", f"{removed} ligne(s) retiree(s)")

    rename_map = {
        "unit": "unite",
        "date": "annee",
        "value": "valeur",
        "indicateurs": "indicateur",
        "pays": "pays",
        "region": "region",
        "milieu": "milieu",
        "sexe": "sexe"
    }
    data = data.rename(columns=rename_map)

    if "indicateur" not in data.columns:
        data["indicateur"] = pd.NA

    categories = [c for c in data.columns if c not in {"indicateur", "unite", "annee", "valeur", "pays", "region", "milieu", "sexe"}]
    if len(categories) >= 1:
        data["categorie_1"] = data[categories[0]]
    else:
        data["categorie_1"] = pd.NA
    if len(categories) >= 2:
        data["categorie_2"] = data[categories[1]]
    else:
        data["categorie_2"] = pd.NA

    for col in ["indicateur", "unite", "pays", "milieu", "sexe", "categorie_1", "categorie_2"]:
        if col in data.columns:
            data[col] = data[col].apply(normalize_text)
    if "region" in data.columns:
        data["region"] = data["region"].apply(clean_geo_name)

    data["annee"] = pd.to_numeric(data.get("annee"), errors="coerce")
    data["valeur"] = pd.to_numeric(data.get("valeur"), errors="coerce")

    out = pd.DataFrame({
        "domaine": "Sante",
        "sous_domaine": sous_domaine,
        "source_file": source_file,
        "indicateur": data.get("indicateur"),
        "categorie_1": data.get("categorie_1"),
        "categorie_2": data.get("categorie_2"),
        "unite": data.get("unite"),
        "annee": data.get("annee"),
        "valeur": data.get("valeur"),
        "pays": data.get("pays"),
        "region": data.get("region"),
        "milieu": data.get("milieu"),
        "sexe": data.get("sexe")
    })

    if "pays" in out.columns:
        out["pays"] = out["pays"].fillna("Burkina Faso")
    out["valeur"] = out["valeur"].fillna(0)

    log_action(sous_domaine, "uniformiser_colonnes_et_types", "schema standard + annee/valeur numeriques")
    return out[KEEP_COLUMNS].copy()# Normalisation francaise appliquee au moment des exportsCOLONNES_FR_MAP = {    "source_file": "fichier_source",    "source_dataset": "jeu_source",    "event_id": "id_evenement",    "country": "pays",    "year": "annee",    "location": "localisation",    "event_type": "type_evenement",    "sub_event_type": "sous_type_evenement",    "event_count": "nombre_evenements",    "fatalities_total": "deces_totaux",    "fatalities_civilians": "deces_civils",    "event_date": "date_evenement",    "granularity": "granularite",}VALEURS_FR_MAP = {    "week": "hebdomadaire",    "event": "evenement",    "acled_weekly_aggregated": "acled_agrege_hebdomadaire",    "ucdp_like_conflict_events": "evenements_conflits_type_ucdp",}def normaliser_dataframe_fr(df: pd.DataFrame) -> pd.DataFrame:    out = df.copy()    out = out.rename(columns={c: COLONNES_FR_MAP.get(c, c) for c in out.columns})    for c in out.columns:        if out[c].dtype == "object":            out[c] = out[c].map(lambda v: VALEURS_FR_MAP.get(str(v).strip().lower(), v) if pd.notna(v) else v)    return out# Renforcement: traduction des valeurs de cellules (expressions + mots) vers le francaisPHRASES_REMPLACEMENTS_FR = {    "individuals using the internet": "Utilisateurs d internet",    "percentage of individuals using the internet": "Utilisateurs d internet",    "internet users": "Utilisateurs d internet",    "mobile cellular subscriptions": "Abonnements de telephonie mobile",    "mobile cellular telephone subscriptions": "Abonnements de telephonie mobile",    "fixed telephone subscriptions": "Abonnements de telephonie fixe",    "fixed telephone lines": "Abonnements de telephonie fixe",    "active mobile broadband subscriptions": "Abonnements actifs internet mobile",    "fixed broadband subscriptions": "Abonnements internet fixe",    "international internet bandwidth": "Bande passante internet internationale",    "acled_weekly_aggregated": "acled agrege hebdomadaire",    "ucdp_like_conflict_events": "evenements conflits type ucdp",}MOTS_REMPLACEMENTS_FR = {    "weekly": "hebdomadaire",    "annual": "annuel",    "event": "evenement",    "events": "evenements",    "conflict": "conflit",    "fatalities": "deces",    "fatality": "deces",    "civilian": "civil",    "civilians": "civils",    "reported": "rapporte",    "targeting": "ciblant",    "number": "nombre",    "country": "pays",    "rate": "taux",    "incidence": "incidence",    "mobile": "mobile",    "fixed": "fixe",    "broadband": "internet",    "subscriptions": "abonnements",    "subscription": "abonnement",    "data": "donnees",    "public": "public",    "private": "prive",}def _traduire_texte_fr(v):    if pd.isna(v):        return v    txt = str(v)    low = txt.lower().strip()    if low in PHRASES_REMPLACEMENTS_FR:        return PHRASES_REMPLACEMENTS_FR[low]    out = txt    for en, fr in MOTS_REMPLACEMENTS_FR.items():        out = re.sub(rf"\b{re.escape(en)}\b", fr, out, flags=re.IGNORECASE)    out = re.sub(r"\s+", " ", out).strip()    return outdef normaliser_dataframe_fr(df: pd.DataFrame) -> pd.DataFrame:    out = df.copy()    out = out.rename(columns={c: COLONNES_FR_MAP.get(c, c) for c in out.columns})    for c in out.columns:        if out[c].dtype == "object":            out[c] = out[c].map(_traduire_texte_fr)    return out

In [ ]:
# 1) Sous-domaine Couverture
couv_dir = RAW_DIR / "couverture_sanitaire"
couv_frames = []
for p in sorted(couv_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not couv_frames:
        print("Apercu brut Couverture:")
        display(raw.head(3))
    couv_frames.append(build_standard_table(raw, "Couverture", p.name))

couverture_clean = pd.concat(couv_frames, ignore_index=True)
print("Apercu nettoye Couverture:")
display(couverture_clean.head(8))
print("Describe Couverture:")
display(couverture_clean[["annee", "valeur"]].describe().T)
print("isnull Couverture:")
display(couverture_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 2) Sous-domaine Epidemiologie
epi_dir = RAW_DIR / "indicateurs_epidemiologiques"
epi_frames = []
for p in sorted(epi_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not epi_frames:
        print("Apercu brut Epidemiologie:")
        display(raw.head(3))
    epi_frames.append(build_standard_table(raw, "Epidemiologie", p.name))

epidemiologie_clean = pd.concat(epi_frames, ignore_index=True)
print("Apercu nettoye Epidemiologie:")
display(epidemiologie_clean.head(8))
print("Describe Epidemiologie:")
display(epidemiologie_clean[["annee", "valeur"]].describe().T)
print("isnull Epidemiologie:")
display(epidemiologie_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 3) Sous-domaine Infrastructures
infra_dir = RAW_DIR / "infrastructures_sanitaires"
infra_frames = []
for p in sorted(infra_dir.glob("*.csv")):
    if p.name.startswith(".~lock"):
        continue
    raw = pd.read_csv(p)
    if not infra_frames:
        print("Apercu brut Infrastructures:")
        display(raw.head(3))
    infra_frames.append(build_standard_table(raw, "Infrastructures", p.name))

infrastructures_clean = pd.concat(infra_frames, ignore_index=True)
print("Apercu nettoye Infrastructures:")
display(infrastructures_clean.head(8))
print("Describe Infrastructures:")
display(infrastructures_clean[["annee", "valeur"]].describe().T)
print("isnull Infrastructures:")
display(infrastructures_clean.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# 4) Indicateurs nationaux (sante_afristat)
sante_afristat_raw = pd.read_csv(RAW_DIR / "sante_afristat.csv")
print("Apercu brut sante_afristat:")
display(sante_afristat_raw.head(3))

sante_afristat_clean = build_standard_table(sante_afristat_raw, "National", "sante_afristat.csv")
print("Apercu nettoye sante_afristat:")
display(sante_afristat_clean.head(8))
print("Describe sante_afristat:")
display(sante_afristat_clean[["annee", "valeur"]].describe().T)
print("isnull sante_afristat:")
display(sante_afristat_clean.isnull().sum().to_frame("nb_manquants"))

## Fusion des datasets necessaires

Fusion annuelle des sous-domaines sur la cle annee afin de disposer d'un tableau comparatif unique (niveau macro).

In [ ]:
annual_couv = couverture_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_couverture"})
annual_epi = epidemiologie_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_epidemiologie"})
annual_infra = infrastructures_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_infrastructures"})
annual_nat = sante_afristat_clean.groupby("annee", as_index=False)["valeur"].sum().rename(columns={"valeur": "valeur_nationale"})

fusion_annuelle = annual_couv.merge(annual_epi, on="annee", how="outer").merge(
    annual_infra, on="annee", how="outer"
).merge(annual_nat, on="annee", how="outer")
for c in ["valeur_couverture", "valeur_epidemiologie", "valeur_infrastructures", "valeur_nationale"]:
    fusion_annuelle[c] = pd.to_numeric(fusion_annuelle[c], errors="coerce").fillna(0)

log_action("fusion_datasets", "merge_annee", "fusion annuelle couverture/epidemiologie/infrastructures/national")

print("Apercu fusion annuelle:")
display(fusion_annuelle.sort_values("annee").head(15))
print("Describe fusion annuelle:")
display(fusion_annuelle.describe().T)
print("isnull fusion annuelle:")
display(fusion_annuelle.isnull().sum().to_frame("nb_manquants"))

In [ ]:
# Harmonisation finale: concatenation des tables standards puis suppression colonnes non importantes
sante_harmonisee_full = pd.concat(
    [couverture_clean, epidemiologie_clean, infrastructures_clean, sante_afristat_clean],
    ignore_index=True
)

dropped_columns = sorted(set(sante_harmonisee_full.columns) - set(KEEP_COLUMNS))
sante_harmonisee = sante_harmonisee_full[KEEP_COLUMNS].copy()
sante_harmonisee = sante_harmonisee.sort_values(["sous_domaine", "annee", "indicateur"], na_position="last")

log_action("harmonisation_finale", "supprimer_colonnes_inutiles", f"colonnes retirees: {dropped_columns}")

print("Colonnes supprimees:")
display(pd.DataFrame({"colonnes_supprimees": dropped_columns}))
print("Apercu table sante_harmonisee:")
display(sante_harmonisee.head(12))
print("Describe sante_harmonisee:")
display(sante_harmonisee[["annee", "valeur"]].describe().T)
print("isnull sante_harmonisee:")
display(sante_harmonisee.isnull().sum().to_frame("nb_manquants"))
display((sante_harmonisee.isnull().mean() * 100).round(2).to_frame("pct_manquants"))

In [ ]:
# Exports
normaliser_dataframe_fr(couverture_clean).to_csv(OUT_COUV / "sante_couverture_harmonisee.csv", index=False)
normaliser_dataframe_fr(epidemiologie_clean).to_csv(OUT_EPI / "sante_epidemiologie_harmonisee.csv", index=False)
normaliser_dataframe_fr(infrastructures_clean).to_csv(OUT_INFRA / "sante_infrastructures_harmonisee.csv", index=False)
normaliser_dataframe_fr(sante_afristat_clean).to_csv(OUT_ROOT / "sante_national_harmonise.csv", index=False)
normaliser_dataframe_fr(fusion_annuelle).to_csv(OUT_ROOT / "sante_fusion_annuelle.csv", index=False)
normaliser_dataframe_fr(sante_harmonisee).to_csv(OUT_ROOT / "sante_harmonisee_global.csv", index=False)

actions_df = pd.DataFrame(action_logs)
print("Journal des actions:")
display(actions_df)

summary = {
    "domaine": "Sante",
    "nb_lignes_couverture": int(len(couverture_clean)),
    "nb_lignes_epidemiologie": int(len(epidemiologie_clean)),
    "nb_lignes_infrastructures": int(len(infrastructures_clean)),
    "nb_lignes_national": int(len(sante_afristat_clean)),
    "nb_lignes_harmonise_global": int(len(sante_harmonisee)),
    "fichiers_generes": [
        "data/Sante/Couverture/sante_couverture_harmonisee.csv",
        "data/Sante/Epidemiologie/sante_epidemiologie_harmonisee.csv",
        "data/Sante/Infrastructures/sante_infrastructures_harmonisee.csv",
        "data/Sante/sante_national_harmonise.csv",
        "data/Sante/sante_fusion_annuelle.csv",
        "data/Sante/sante_harmonisee_global.csv"
    ]
}

with open(OUT_ROOT / "synthese_preparation_sante.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

## Recapitulatif

Ce notebook nettoie les fichiers bruts du module Sante, standardise les colonnes, corrige les types et les valeurs manquantes, harmonise les sous-domaines, realise une fusion annuelle utile, supprime les colonnes non importantes, affiche les controles en cellules (apercu/describe/isnull), puis exporte les jeux harmonises dans data/Sante.